# Stock Market Prediction Using Machine Learning and Ensemble Learning

## Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import yfinance as yf
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.linear_model import (LinearRegression,Ridge,Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split,TimeSeriesSplit,cross_validate,GridSearchCV)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,r2_score)


sns.set_theme()

## Data Processing


In [2]:
# Ticker = input(print('Enter Ticker: '))
Ticker ='MSFT'

df = yf.download(Ticker,'2020-01-01')

[*********************100%***********************]  1 of 1 completed


In [3]:
df.head()

Price,Close,High,Low,Open,Volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT
Date,,,,,
2020-01-02,151.829575,151.933555,149.664908,150.090278,22622100
2020-01-03,149.938995,151.196208,149.409646,149.655425,21116200
2020-01-06,150.326569,150.392745,147.944480,148.483292,20813700
2020-01-07,148.955948,150.931563,148.710182,150.600726,21634100
2020-01-08,151.328598,151.999748,149.305716,150.232079,27746500


In [4]:
df['Target'] = df['Close'].shift(-1)
df.head()

Price,Close,High,Low,Open,Volume,Target
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,
Date,,,,,,
2020-01-02,151.829575,151.933555,149.664908,150.090278,22622100,149.938995
2020-01-03,149.938995,151.196208,149.409646,149.655425,21116200,150.326569
2020-01-06,150.326569,150.392745,147.944480,148.483292,20813700,148.955948
2020-01-07,148.955948,150.931563,148.710182,150.600726,21634100,151.328598
2020-01-08,151.328598,151.999748,149.305716,150.232079,27746500,153.219101


In [5]:

df['MA_10'] = df['Close'].rolling(10).mean()
df['MA_50'] = df['Close'].rolling(50).mean()


df['Volatility'] = df['Close'].rolling(10).std()


df['Daily_Return'] = df['Close'].pct_change()

In [6]:
df.dropna(axis=0,inplace=True)

In [7]:
df.describe()

Price,Close,High,Low,Open,Volume,Target,MA_10,MA_50,Volatility,Daily_Return
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,,,,,
count,1611.000000,1611.000000,1611.000000,1611.000000,1.611000e+03,1611.000000,1611.000000,1611.000000,1611.000000,1611.000000
mean,328.667522,331.817068,325.264028,328.598099,2.810568e+07,328.879739,327.692871,324.704804,6.447200,0.001007
std,97.899169,98.437069,97.343280,97.985906,1.292116e+07,97.883367,97.680904,98.041844,4.407729,0.018750
min,128.358383,133.239757,125.609535,129.865387,5.855900e+06,128.358383,135.692848,153.334292,0.841681,-0.147390
25%,244.532074,246.754747,241.438232,243.961253,1.994990e+07,244.636482,244.437042,241.105906,3.866779,-0.008082
50%,319.236420,322.457631,316.378303,319.305398,2.513880e+07,319.642700,318.918939,313.413359,5.551165,0.000847
75%,410.134247,413.558983,406.171606,410.429821,3.266555e+07,410.168839,411.436836,409.659534,7.654963,0.010485
max,538.658569,551.048474,537.366763,550.830186,1.862016e+08,538.658569,522.501892,511.207405,49.547926,0.155067


In [8]:
df.isnull().sum()


Price         Ticker
Close         MSFT      0
High          MSFT      0
Low           MSFT      0
Open          MSFT      0
Volume        MSFT      0
Target                  0
MA_10                   0
MA_50                   0
Volatility              0
Daily_Return            0
dtype: int64

## Feature Selection

In [9]:
target = df['Target']
features = df[['Close','Volume','High','Low','Open',
         'MA_10','MA_50',
         'Volatility']]

## Train Test Split

In [10]:
x_train,x_test,y_train,y_test = train_test_split(features,target,test_size = 0.2,shuffle = False)

## Cross Validation

In [11]:
# TimeSeries Split
tscv = TimeSeriesSplit(n_splits=5)

scoring = {
    'R2':'r2',
    'MAE':'neg_mean_absolute_error',
    'RMSE':'neg_root_mean_squared_error'
}

def run_cross_validation(model,x,y):
    scores = cross_validate(
        model,
        x,
        y,
        cv = tscv,
        scoring = scoring
    )

    print(f"Average R2 : {scores["test_R2"].mean():.2f}")
    print(f"Average MAE : {-scores["test_MAE"].mean():.2f}")
    print(f"Average RMSE : {-scores["test_RMSE"].mean():.2f}")


# Model Selection

## Linear Regression

In [12]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

run_cross_validation(lr,x_train,y_train)

Average R2 : 0.96
Average MAE : 4.14
Average RMSE : 5.41


## Ridge Regression

In [13]:
ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

run_cross_validation(ridge,x_train,y_train)

Average R2 : 0.96
Average MAE : 4.28
Average RMSE : 5.56


## Lasso Regression


In [14]:
lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(max_iter=10000))
])

run_cross_validation(lasso,x_train,y_train)

Average R2 : 0.95
Average MAE : 4.96
Average RMSE : 6.21


##  Random Forest

In [15]:
rf = RandomForestRegressor(
    random_state=42,
)

run_cross_validation(rf,x_train,y_train)

Average R2 : -0.10
Average MAE : 21.73
Average RMSE : 26.58


## XG Boost Regressor

In [16]:
xgb = XGBRegressor(
    random_state=42,
)

run_cross_validation(xgb,x_train,y_train)

Average R2 : -0.32
Average MAE : 24.03
Average RMSE : 28.99


## Hyperparameter Tuning with GridSearchCV For Linear Regression

In [17]:
param_grid = {
    'model__fit_intercept': [True, False],
    'model__positive': [True, False]
}

grid_search = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    n_jobs=-1
)

grid_search.fit(x_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV RMSE:")
print(-grid_search.best_score_)

Best Parameters:
{'model__fit_intercept': True, 'model__positive': True}
Best CV RMSE:
5.304863688067096


In [18]:
best_lr = grid_search.best_estimator_

run_cross_validation(
    best_lr,
    x_train,
    y_train
)

Average R2 : 0.96
Average MAE : 4.06
Average RMSE : 5.30


In [19]:
joblib.dump(best_lr,"../model/model.pkl")

['../model/model.pkl']

## Training Best Model (Linear Regression)

In [20]:
best_lr.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](8,)","[ 297.27,27857275.08, 300.14,..., 296.47, 293.14, 5.66]"


In [21]:
y_pred = best_lr.predict(x_test)

## Evaluation

In [22]:
test_r2 = r2_score(y_test,y_pred)
test_mae = mean_absolute_error(y_test,y_pred)
test_mse = mean_squared_error(y_test,y_pred)
test_rmse = np.sqrt(test_mse)

print(f"R2 : {test_r2:.2f}")
print(f"MAE : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")

R2 : 0.97
MAE : 5.50
RMSE : 8.24


In [23]:
naive_pred = x_test['Close'].values

naive_mae = mean_absolute_error(y_test, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))

print(f"Naive MAE  : {naive_mae:.2f}")
print(f"Naive RMSE : {naive_rmse:.2f}")

Naive MAE  : 5.39
Naive RMSE : 8.11
